### Simulation of Chosen stocks overtime based different simulation types

- Geometric Brownian Simulation
- Heston Model
- Merton Jump-Diffusion Model


In [ ]:
#| include: false
%matplotlib inline
import numpy as np
import pandas as pd
import yfinance as yf 
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.widgets import Button
import matplotlib as mpl
from IPython.display import HTML, display
from scipy.stats import skew, kurtosis, ttest_rel, wilcoxon, ks_2samp, wasserstein_distance
from statsmodels.stats.multitest import multipletests

### Geometric Brownian Motion (GBM)
The main formula for the Geometric Brownian Motion (GBM) is:
$$ dS_t = \mu S_t dt + \sigma S_t dW_t$$
Where: $S_t$    = Stock Price at time t
       $\mu$    = expected annual return (drift)
       $\sigma$ = annual volatility
       $dt$     = small change in time
       $dW_t$.  = Brownian motion/random market shock

For our Monte Carlo Simulation our Equation becomes
$$ S_{t+\Delta} = S_t exp[(\mu-\frac{1}{2}\sigma^2) \Delta t + \sigma \sqrt{ \Delta t }Z]$$

Where $$ Z \sim N(0,1) $$

In [ ]:
#| echo: false
def GBM(IAP, MU, SIGMA, T, n, Paths):
    dt = T / n
    #Variables
    drift = (MU - 0.5 * SIGMA**2) * dt
    diffusion = SIGMA * np.sqrt(dt)
    Z = np.random.standard_normal(size = (n, Paths))
    #Variables combined
    Path_mult = np.exp(drift + diffusion * Z)
    #Added array of 1's to be replaced with real values by number of n 
    array_fill = np.ones((1, Paths))
    #variable combination into array
    array_full = np.vstack([array_fill, Path_mult])
    paths = IAP * np.cumprod(array_full, axis = 0)

    return paths

#Simulation Parameters   
np.random.seed(43)  
time_horizon = 1.0       
steps = 252            
simulations = 1000  

Stock = "AIR.NZ"

stock_data = yf.download(Stock, period="10y", progress=False)

if isinstance(stock_data.columns, pd.MultiIndex):
    close_prices = stock_data['Close'][Stock]
else:
    close_prices = stock_data['Close']

close_prices = close_prices.dropna()

IAP_stock = float(close_prices.iloc[-1])
log_returns = np.log(close_prices / close_prices.shift(1)).dropna()

daily_mean = log_returns.mean()
daily_var = log_returns.var()
mu = (daily_mean + 0.5 * daily_var) * steps

sigma = log_returns.std() * np.sqrt(steps)

print("\n--- Model Calibration Parameters ---")
print(f"Latest Stock Price (IAP) : ${IAP_stock:.2f}")
print(f"Calculated Annual Drift  : {mu:.4%}")
print(f"Calculated Volatility    : {sigma:.4%}")

#Simulation
gbm_paths = GBM(IAP_stock, mu, sigma, time_horizon, steps, simulations)

#Plot
time_grid = np.linspace(0, time_horizon, steps+1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
plt.subplots_adjust(bottom=0.2)

#Time Series plot
ax1.set_xlim(0, time_horizon)
ax1.set_ylim(np.min(gbm_paths) * 0.9, np.max(gbm_paths) * 1.1)
ax1.set_title(f"Live Monte Carlo Paths ({Stock})")
ax1.set_xlabel("Time (Years)")
ax1.set_ylabel("Asset Price ($)")
ax1.grid(True, linestyle="--", alpha=0.3)

lines = [ax1.plot([], [], linewidth = 1, alpha = 0.5)[0] for _ in range(simulations)]

#Distribution
ax2.set_xlim(np.min(gbm_paths[-1, :]) * 0.9, np.max(gbm_paths[-1, :]) * 1.1)
ax2.set_title("Live Distribution Counts")
ax2.set_xlabel("Ending Price ($)")
ax2.set_ylabel("Count")
ax2.grid(True, linestyle="--", alpha=0.3)

start_line = ax2.axvline(IAP_stock, color='red', linestyle='--', linewidth=2, label=f"Start (${round(IAP_stock, 2)})")
median_line = ax2.axvline(0, color='green', linewidth = 2, label = "Current Median")
ax2.legend(loc='upper right')

#Animation Loop/live simulation
display_paths = min(simulations, 200)
lines = [
    ax1.plot([], [], linewidth=0.8, alpha=0.5)[0]
    for _ in range(display_paths)
]

def update(frame):
    for i, line in enumerate(lines):
        line.set_data(
            time_grid[:frame + 1],
            gbm_paths[:frame + 1, i]  
        )

    for patch in list(ax2.patches):
        patch.remove()

    current_prices = gbm_paths[frame, :]  
    counts, _ = np.histogram(current_prices, bins=20)

    current_median = np.median(current_prices)
    median_line.set_xdata([current_median, current_median])
    median_line.set_label(f"Median (${current_median:.2f})")
    ax2.legend(loc="upper right")
    ax2.set_ylim(0, max(counts) * 1.2 if max(counts) else 10)

    ax2.hist(
        current_prices,
        bins=20,
        color="royalblue",
        edgecolor="black",
        alpha=0.7
    )

    return lines + [median_line]

ani = FuncAnimation(
    fig,
    update,
    frames=steps + 1,
    interval=30,
    blit=False,
    repeat=False,
    cache_frame_data=False
)

mpl.rcParams["animation.embed_limit"] = 100

update(steps)
display(fig)
plt.close(fig)


1. Limiation of the GBM model is historical calibration. Stocks such as MU have very large volatility over a very short period of time. This leads to very large calculated annual drift and volatility for the simulation, leading to very large swings in the simulation and long right tails. The median does seem reasonible compared to the Heston however. A way to deal with this is putting a cap but this is less defensible as it forces the drift into a range to make it more reasonable then the estimates give. 

In [ ]:
#| eval: false
#| include: false
display(HTML(ani.to_jshtml()))

### Heston Model
The Heston model consists of two stochastic differential equations with a large amount of parameters compared to the GBM
 - Stock-price equation:
 $$ dS_t = \mu S_tdt + \sqrt{V_tS_tdW^S_t} $$
 - Variance equation
 $$ dV_t = \kappa(\theta-V_t)dt + \sigma \sqrt{V_tS_tdW^S_t} $$
With two Brownian motions related by
$$dW_t^S dW_t^V = pdt $$

Where: $S_t$ = Stock price; $V_t$ = instantaneous variance; $\mu$ = expected return/drift; $\kappa$ = speed of mean reversion in variance; $\theta$ - long-run average variance; $\sigma$ = volatility of variance, or vol-of-vol; $p$ = correlation between stock-price and variance shocks; $dW^S_t$ = Stock-price Brownian shock; and $dW^V_t $ = Variance Brownian shock
 
 For the simulation, the stock-price equation becomes:
 $$S_{t+\Delta t} = S_t~exp[(\mu-\frac{1}{2}V_t)\Delta t + \sqrt{V_t \Delta t} Z^S_t] $$
and the variance equation becomes: 
$$ V_{t+\Delta t} = V_t + \kappa(\theta-V_t)\Delta t + \sigma \sqrt{V_t\Delta t}Z^V_t$$
With 
$$ Z^V_t = pZ^S_t + \sqrt{1-p^2}Z^2_t $$
Where $Z^s_t$ and $Z^2_t$ are independent standard normal random variables

In [ ]:
#| echo: false
def Heston(IAP, V0, r, kappa, theta, sigma, rho, T, n, sims):
    dt = T / n
    S = np.zeros((n + 1, sims))
    V = np.zeros((n + 1, sims))
    S[0] = IAP
    V[0] = V0

    for t in range(1, n + 1):
        Z1 = np.random.normal(0,1,sims)
        Z2 = np.random.normal(0,1,sims)
        W_S = Z1
        W_V = rho * Z1 + np.sqrt(1 - rho**2) * Z2

        V_prev_truncated = np.maximum(V[t-1], 0)

        V[t] = np.maximum(
            V[t-1]
            + kappa * (theta - V_prev_truncated) * dt
            + sigma * np.sqrt(V_prev_truncated * dt) * W_V,
            0)
        S[t] = S[t-1] * np.exp(
            (mu - 0.5 * V_prev_truncated) * dt
            + np.sqrt(V_prev_truncated * dt) * W_S
)
    return S, V

np.random.seed(43)   
time_horizon = 1.0     
steps = 252        
simulations = 1000  

Stock = "AIR.NZ"


stock_data = yf.download(Stock, period="10y", progress= False)

if isinstance(stock_data.columns, pd.MultiIndex):
    close_prices = stock_data['Close'][Stock]
else:
    close_prices = stock_data['Close']
close_prices = close_prices.dropna()


IAP_stock = float(close_prices.iloc[-1])
log_returns = np.log(close_prices / close_prices.shift(1)).dropna()

daily_mean = log_returns.mean()
daily_var = log_returns.var()

mu = (daily_mean + 0.5 * daily_var) * steps

rolling_variance = (
    log_returns
    .rolling(window=21)
    .var()
    * steps
).dropna()

V0 = float(rolling_variance.iloc[-1])

#Theta and Kappa Estimates
V_t = rolling_variance.iloc[:-1].values
V_next = rolling_variance.iloc[1:].values

delta_V = V_next - V_t

X = np.column_stack([
    np.ones(len(V_t)),
    V_t
])

beta = np.linalg.lstsq(
    X,
    delta_V,
    rcond=None
)[0]
dt = 1 / steps
intercept = beta[1]
slope = beta[1]
kappa = -slope / dt
kappa = max(float(kappa), 1e-6)
theta = intercept / (kappa * dt)


# Sigma Estimates
if not np.isfinite(theta) or theta <= 0:
    theta = float(rolling_variance.mean())

expected_delta_V = (
    kappa* (theta - V_t)
    * dt
)

variance_residuals = (
    delta_V - expected_delta_V
)

valid = V_t > 1e-10

standardized_variance_shocks = (
    variance_residuals[valid]
    / np.sqrt(V_t[valid] * dt)
)

sigma = float(
    np.std(
        standardized_variance_shocks,
        ddof=1
    )
)

if not np.isfinite(sigma) or sigma <= 0:
    sigma = 0.01

#rho Estimate 
aligned_returns = (
    log_returns
    .loc[rolling_variance.index]
    .iloc[1:]
    .values
)
aligned_returns = aligned_returns[valid]
V_for_returns = V_t[valid]

stock_shocks = (
    aligned_returns
    - mu * dt
) / np.sqrt(
    V_for_returns * dt
)

variance_shocks = (
    variance_residuals[valid]
    / (
        sigma
        * np.sqrt(V_for_returns * dt)
    )
)

rho = float(
    np.corrcoef(
        stock_shocks,
        variance_shocks
    )[0, 1]
)

if not np.isfinite(rho):
    rho = 0.0

rho = np.clip(
    rho,
    -0.999,
    0.999
)

#Printed Estiamtes for each stock
print("\n--- Model Calibration Parameters ---")
print(f"Latest Stock Price (IAP) : ${IAP_stock:.2f}")
print(f"Calculated mu value  : {mu:.4%}")
print(f"Calculated V0 Value : {V0:.4%}")
print(f"Calculated theta Value : {theta:.4%}")
print(f"Calculated kappa Value : {kappa:.4}")
print(f"Calculated rho Value : {rho:.4}")
print(f"Calculated sigma Value : {sigma:.4}")
print(
    f"Feller condition         : "
    f"{'Satisfied' if 2 * kappa * theta > sigma**2 else 'Not satisfied'}"
)

#Simulation
heston_prices, heston_variances = Heston(IAP_stock, V0, mu, kappa, theta, sigma, rho, time_horizon, steps, simulations)
Heston_paths = heston_prices

#Plot
time_grid = np.linspace(0, time_horizon, steps+1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
plt.subplots_adjust(bottom=0.25)

#Time Series plot
ax1.set_xlim(0, time_horizon)
ax1.set_ylim(np.min(Heston_paths) * 0.9, np.max(Heston_paths) * 1.1)
ax1.set_title(f"Live Monte Carlo Paths ({Stock})")
ax1.set_xlabel("Time (Years)")
ax1.set_ylabel("Asset Price ($)")
ax1.grid(True, linestyle="--", alpha=0.3)

lines = [ax1.plot([], [], linewidth = 1, alpha = 0.5)[0] for _ in range(simulations)]

#Distribution
ax2.set_xlim(np.min(Heston_paths[-1, :]) * 0.9, np.max(Heston_paths[-1, :]) * 1.1)
ax2.set_title("Live Distribution Counts") 
ax2.set_xlabel("Ending Price ($)")
ax2.set_ylabel("Count")
ax2.grid(True, linestyle="--", alpha=0.3)

start_line = ax2.axvline(IAP_stock, color='red', linestyle='--', linewidth=2, label=f"Start (${round(IAP_stock, 2)})")
median_line = ax2.axvline(0, color='green', linewidth = 2, label = f"Current Median")
ax2.legend(loc='upper right')

from IPython.display import HTML, display

# Render fewer lines; keep all simulations for the histogram
display_paths = min(simulations, 200)
lines = [
    ax1.plot([], [], linewidth=0.8, alpha=0.5)[0]
    for _ in range(display_paths)
]

def update(frame):
    for i, line in enumerate(lines):
        line.set_data(
            time_grid[:frame + 1],
            Heston_paths[:frame + 1, i]  
        )

    for patch in list(ax2.patches):
        patch.remove()

    current_prices = Heston_paths[frame, :]  
    counts, _ = np.histogram(current_prices, bins=20)
    current_median = np.median(current_prices)

    median_line.set_xdata([current_median, current_median])
    median_line.set_label(f"Median (${current_median:.2f})")
    ax2.legend(loc="upper right")
    ax2.set_ylim(0, max(counts) * 1.2 if max(counts) else 10)
    ax2.hist(
        current_prices,
        bins=20,
        color="royalblue",
        edgecolor="black",
        alpha=0.7
    )

    return lines + [median_line]

ani = FuncAnimation(
    fig,
    update,
    frames=steps + 1,
    interval=30,
    blit=False,
    repeat=False,
    cache_frame_data=False
)

mpl.rcParams["animation.embed_limit"] = 100

update(steps)
display(fig)
plt.close(fig)

In [ ]:
#| eval: false
#| include: false
#| warnings: false
display(HTML(ani.to_jshtml()))

### Comparison of models

In [ ]:
#| echo: false
np.random.seed(43)
stocks = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL",
    "META", "DORM", "GS", "BNY", "REGN",
    "LRCX", "GEV", "KO", "PEP", "WMT",
    "COST", "CVX", "CAT", "GE", "HWM"
]

time_horizon = 1
steps = 252
simulations = 10000

#GBM Model
def GBM(IAP, MU, gbm_sigma, T, n, Paths):
    dt = T / n
    #Variables
    drift = (MU - 0.5 * gbm_sigma**2) * dt
    diffusion = gbm_sigma * np.sqrt(dt)
    Z = np.random.standard_normal(size = (n, Paths))
    #Variables combined
    Path_mult = np.exp(drift + diffusion * Z)
    #Added array of 1's to be replaced with real values by number of n 
    array_fill = np.ones((1, Paths))
    #variable combination into array
    array_full = np.vstack([array_fill, Path_mult])
    paths = IAP * np.cumprod(array_full, axis = 0)

    return paths

#Heston Model
def Heston(IAP, V0, r, kappa, theta, heston_sigma, rho, T, n, sims):
    dt = T / n
    S = np.zeros((n + 1, sims))
    V = np.zeros((n + 1, sims))
    S[0] = IAP
    V[0] = V0

    for t in range(1, n + 1):
        Z1 = np.random.normal(0,1,sims)
        Z2 = np.random.normal(0,1,sims)
        W_S = Z1
        W_V = rho * Z1 + np.sqrt(1 - rho**2) * Z2

        V_prev_truncated = np.maximum(V[t-1], 0)

        V[t] = np.maximum(
            V[t-1]
            + kappa * (theta - V_prev_truncated) * dt
            + heston_sigma * np.sqrt(V_prev_truncated * dt) * W_V,
            0)
        S[t] = S[t-1] * np.exp(
            (mu - 0.5 * V_prev_truncated) * dt
            + np.sqrt(V_prev_truncated * dt) * W_S
)
    return S, V

results = []

# Stocks Loop

for Stock in stocks:

    print(f"Running models for {Stock}...")

    stock_data = yf.download(
        Stock,
        period="5y",
        progress=False
    )
    if isinstance(stock_data.columns, pd.MultiIndex):
        close_prices = stock_data['Close'][Stock]
    else:
        close_prices = stock_data['Close']
    close_prices = close_prices.dropna()

    if len(close_prices) < 100:
        print(f"Skipping {Stock}: insufficient data")
        continue

#Data Across Models
    IAP_stock = float(close_prices.iloc[-1])
    log_returns = np.log(close_prices / close_prices.shift(1)).dropna()

    daily_mean = log_returns.mean()
    daily_var = log_returns.var()

    mu = (daily_mean + 0.5 * daily_var) * steps

# GBM Parameters

    gbm_sigma = log_returns.std() * np.sqrt(steps)

#Heston Parameters

    rolling_variance = (log_returns.rolling(window=21).var()* steps).dropna()

    V0 = float(rolling_variance.iloc[-1])

#Theta and Kappa Estimates
    V_t = rolling_variance.iloc[:-1].values
    V_next = rolling_variance.iloc[1:].values
    delta_V = V_next - V_t
    X = np.column_stack([np.ones(len(V_t)), V_t])

    beta = np.linalg.lstsq(X, V_next, rcond=None)[0]

    dt = 1 / steps

    intercept = beta[0]
    slope = beta[1]
    


    if 0 < slope < 1:

        kappa = -np.log(slope) / dt

        theta = intercept / (1-slope)

    else:

        theta = float(rolling_variance.mean())

        kappa = 2.0

    if not np.isfinite(theta) or theta <= 0:
        theta = float(rolling_variance.mean())

    if not np.isfinite(kappa) or kappa <= 0:
        kappa = 2.0

# Sigma Estimates
    delta_V = V_next - V_t

    expected_delta_V = (kappa* (theta - V_t)* dt)

    variance_residuals = (delta_V - expected_delta_V)

    valid = V_t > 1e-10

    standardized_variance_shocks = (variance_residuals[valid] 
                                    / np.sqrt(V_t[valid] * dt))

    heston_sigma = float(np.std(standardized_variance_shocks,ddof=1))

    if not np.isfinite(heston_sigma) or heston_sigma <= 0:
        heston_sigma = 0.01

#rho Estimate 
    aligned_returns = (log_returns.loc[rolling_variance.index].iloc[1:].values)
    aligned_returns = aligned_returns[valid]
    V_for_returns = V_t[valid]

    stock_shocks = (aligned_returns - mu * dt) / np.sqrt(V_for_returns * dt)

    variance_shocks = (variance_residuals[valid]/ 
                    (heston_sigma * np.sqrt(V_for_returns * dt)))

    rho = float(np.corrcoef(stock_shocks, variance_shocks)[0, 1])

    if not np.isfinite(rho):
        rho = 0.0

    rho = np.clip(rho,-0.999,0.999)

# GBM Sim
    gbm_paths = GBM(IAP_stock, mu, gbm_sigma, time_horizon, steps, simulations)

    gbm_final = (
        gbm_paths[-1, :]
    )

#Heston Sim
    heston_paths, heston_variances = Heston(IAP_stock, V0, mu, kappa, theta, heston_sigma, rho, time_horizon, steps, simulations)

    heston_final = (
    heston_paths[-1, :]
    )

#Stats

    results.append({
#General Stock Parameters
        "Stock": Stock,

        "Start_Price": IAP_stock,

        "Mu": mu,
#GBM Parameters
        "GBM_Sigma": gbm_sigma,
#Heston Parameters
        "Heston_V0": V0,
        "Heston_Theta": theta,
        "Heston_Kappa": kappa,
        "Heston_Sigma": heston_sigma,
        "Heston_Rho": rho,
#GBM Distribution
        "GBM_Mean": np.mean(gbm_final),
        "GBM_Median": np.median(gbm_final),
        "GBM_Std": np.std(gbm_final,ddof=1),

        "GBM_5%": np.percentile(gbm_final,5),
        "GBM_25%": np.percentile(gbm_final,25),
        "GBM_75%": np.percentile(gbm_final,75),
        "GBM_95%": np.percentile(gbm_final,95),

        "GBM_Prob_Loss": np.mean(gbm_final < IAP_stock),
        "GBM_Skewness": skew(gbm_final),
        "GBM_Kurtosis": kurtosis(gbm_final),
# Heston Distribution 
        "Heston_Mean": np.mean(heston_final),
        "Heston_Median": np.median(heston_final),
        "Heston_Std": np.std(heston_final,ddof=1),

        "Heston_5%": np.percentile(heston_final,5),
        "Heston_25%": np.percentile(heston_final,25),
        "Heston_75%": np.percentile(heston_final,75),
        "Heston_95%": np.percentile(heston_final, 95),

        "Heston_Prob_Loss": np.mean(heston_final < IAP_stock),
        "Heston_Skewness": skew(heston_final),
        "Heston_Kurtosis": kurtosis(heston_final)
    })

    print(
        f"Added {Stock}."
        f"Rows stored: {len(results)}"
    )
#Data Frame

results_df = pd.DataFrame(
    results
)

display(results_df)

comparison_df = results_df[[
    "Stock",

    "GBM_Mean",
    "Heston_Mean",

    "GBM_Median",
    "Heston_Median",

    "GBM_Std",
    "Heston_Std",

    "GBM_5%",
    "Heston_5%",

    "GBM_95%",
    "Heston_95%",

    "GBM_Prob_Loss",
    "Heston_Prob_Loss",

    "GBM_Skewness",
    "Heston_Skewness",

    "GBM_Kurtosis",
    "Heston_Kurtosis"
]].copy()

In [ ]:
#| eval: false
#| include: false
#| warnings: false
results_df.to_csv('/Users/connorpiercy/Stock Simulation/Stock-Simulations\results.csv', index=False)

### Comparing Models
Too compare the models we will be using two difference tests; A Paired t-test and a Wilcoxon test. For each stock, the difference between the Heston and GBM value was calculated for each distributional statistic. Paired tests were then applied across the 20 stock-level differences to determine whether either model systematically produced higher or lower values for the statistic.

### Paired t-test of model differences
$$t = \frac{\hat{d}}{\frac{S_d}{\sqrt{n}}},~t \sim t_{n-1}$$
Where $\hat{d}$ is the sample difference between paired means, $S_d$ os the sample standard deviation of differences, and $n$ is the number of pairs.
$$Null~Hypothesis: \mu_d=0$$
$$Alternative~Hypothesis: \mu_d\neq 0$$

In [ ]:
#| echo: false

#Median
comparison_df["GBM_Median_Return"] = (
    comparison_df["GBM_Median"]
    / results_df["Start_Price"]
    - 1
)
comparison_df["Heston_Median_Return"] = (
    comparison_df["Heston_Median"]
    / results_df["Start_Price"]
    - 1
)
#SD
comparison_df["GBM_Relative_Std"] = (
    comparison_df["GBM_Std"]
    / results_df["Start_Price"]
)
comparison_df["Heston_Relative_Std"] = (
    comparison_df["Heston_Std"]
    / results_df["Start_Price"]
)
#Lower Tail
comparison_df["GBM_5_Return"] = (
    comparison_df["GBM_5%"]
    / results_df["Start_Price"]
    - 1
)
comparison_df["Heston_5_Return"] = (
    comparison_df["Heston_5%"]
    / results_df["Start_Price"]
    - 1
)
#Upper Tail
comparison_df["GBM_95_Return"] = (
    comparison_df["GBM_95%"]
    / results_df["Start_Price"]
    - 1
)
comparison_df["Heston_95_Return"] = (
    comparison_df["Heston_95%"]
    / results_df["Start_Price"]
    - 1
)
#Prob Loss
comparison_df["Prob_Loss_Diff"] = (
    comparison_df["Heston_Prob_Loss"]
    - comparison_df["GBM_Prob_Loss"]
)
#Skew
comparison_df["Skewness_Diff"] = (
    comparison_df["Heston_Skewness"]
    - comparison_df["GBM_Skewness"]
)
#Kurtosis
comparison_df["Kurtosis_Diff"] = (
    comparison_df["Heston_Kurtosis"]
    - comparison_df["GBM_Kurtosis"]
)

display(comparison_df)

In [ ]:
#| echo: false

metrics = [
    "Median_Return",
    "Relative_Std",
    "5_Return",
    "95_Return",
    "Prob_Loss",
    "Skewness",
    "Kurtosis"
]

test_results = []

for metric in metrics:

    gbm = comparison_df[f"GBM_{metric}"]
    heston = comparison_df[f"Heston_{metric}"]

    differences = heston - gbm

    t_stat, t_p = ttest_rel(heston, gbm)
    w_stat, w_p = wilcoxon(heston, gbm)

    test_results.append({
        "Metric": metric,
        "GBM_Average": gbm.mean(),
        "Heston_Average": heston.mean(),
        "Average_Difference": differences.mean(),
        "T_p_value": t_p,
        "Wilcoxon_p_value": w_p
    })

test_df = pd.DataFrame(test_results)

display(test_df)

#Holm Correction for pair t-test p-values

test_df["T_p_Holm"] = multipletests(
    test_df["T_p_value"],
    alpha=0.05,
    method="holm"
)[1]

test_df["Wilcoxon_p_Holm"] = multipletests(
    test_df["Wilcoxon_p_value"],
    alpha=0.05,
    method="holm"
)[1]

test_df["T_Holm_Significant"] = (
    test_df["T_p_Holm"] < 0.05
)

test_df["Wilcoxon_Holm_Significant"] = (
    test_df["Wilcoxon_p_Holm"] < 0.05
)

display(test_df)

- Across the 20 equities, GBM and Heston produced statistically similar median returns, upper-tail returns, and probabilities of loss. However, Heston produced significantly lower 5th-percentile returns, greater positive skewness, and substantially greater kurtosis. This indicates that introducing stochastic volatility had relatively little effect on the centre of the simulated terminal-price distributions but materially altered their tail behaviour and distributional shape. 

- However, after applying the Holm correction for multiple comparisons, statistically significant differences remained only for the 5th-percentile return and kurtosis. The Heston model produced a significantly lower 5th-percentile return than GBM, indicating greater downside-tail risk, and significantly higher kurtosis, indicating heavier tails. Differences in median return, relative standard deviation, 95th-percentile return, probability of loss, and skewness were not statistically significant after correction.

- This result supports theory behind the Heston model. The model doesnt shift the centre of the forecasted distribution as seen in the results, but changes the behaviour of the tail instead. 

### Two-Sample Kolmogorov-Smirnov Test:

This test is to measure the maximum vertical distance between the empirical cumulative distribution of the GBM and Heston Simulations for each stock.

K-S Statistic is
$$D=\underset{x}{sup}[F_{GBM}(x)-F_{Heston}(x)].$$
with Hypothesis:
$$H_0:F_{GBM}(x)=F_{Heston}(x); H_1:F_{GBM}(x) \neq F_{Heston}(x).$$

Due to having 10,000 simulations per model per stock, the K-S p-value may become tiny for relatively small differences. Therefore for interpretation we will emphasize the K-S statistic $D$ as the effect size, not only whether $p < 0.05$.

A Holm Correction will be added to the p-values obtained to adjust for the fact we are doing multiple tests simultaneously.


In [ ]:
#| echo = false

np.random.seed(43)
results = []

for Stock in stocks:

    stock_data = yf.download(
        Stock,
        period="5y",
        progress=False
    )

    if isinstance(stock_data.columns, pd.MultiIndex):
        close_prices = stock_data["Close"][Stock]
    else:
        close_prices = stock_data["Close"]

    close_prices = close_prices.dropna()

    IAP_stock = float(close_prices.iloc[-1])

    log_returns = np.log(
        close_prices / close_prices.shift(1)
    ).dropna()

    # GBM Parameters

    gbm_sigma = log_returns.std() * np.sqrt(steps)

#Heston Parameters

    rolling_variance = (log_returns.rolling(window=21).var()* steps).dropna()

    V0 = float(rolling_variance.iloc[-1])

#Theta and Kappa Estimates
    V_t = rolling_variance.iloc[:-1].values
    V_next = rolling_variance.iloc[1:].values
    delta_V = V_next - V_t
    X = np.column_stack([np.ones(len(V_t)), V_t])

    beta = np.linalg.lstsq(X, V_next, rcond=None)[0]

    dt = 1 / steps

    intercept = beta[0]
    slope = beta[1]
    


    if 0 < slope < 1:

        kappa = -np.log(slope) / dt

        theta = intercept / (1-slope)

    else:

        theta = float(rolling_variance.mean())

        kappa = 2.0

    if not np.isfinite(theta) or theta <= 0:
        theta = float(rolling_variance.mean())

    if not np.isfinite(kappa) or kappa <= 0:
        kappa = 2.0

# Sigma Estimates
    delta_V = V_next - V_t

    expected_delta_V = (kappa* (theta - V_t)* dt)

    variance_residuals = (delta_V - expected_delta_V)

    valid = V_t > 1e-10

    standardized_variance_shocks = (variance_residuals[valid] 
                                    / np.sqrt(V_t[valid] * dt))

    heston_sigma = float(np.std(standardized_variance_shocks,ddof=1))

    if not np.isfinite(heston_sigma) or heston_sigma <= 0:
        heston_sigma = 0.01

#rho Estimate 
    aligned_returns = (log_returns.loc[rolling_variance.index].iloc[1:].values)
    aligned_returns = aligned_returns[valid]
    V_for_returns = V_t[valid]

    stock_shocks = (aligned_returns - mu * dt) / np.sqrt(V_for_returns * dt)

    variance_shocks = (variance_residuals[valid]/ 
                    (heston_sigma * np.sqrt(V_for_returns * dt)))

    rho = float(np.corrcoef(stock_shocks, variance_shocks)[0, 1])

    if not np.isfinite(rho):
        rho = 0.0

    rho = np.clip(rho,-0.999,0.999)

# GBM Sim
    gbm_paths = GBM(IAP_stock, mu, gbm_sigma, time_horizon, steps, simulations)

    gbm_final = (
        gbm_paths[-1, :]
    )

#Heston Sim
    heston_paths, heston_variances = Heston(IAP_stock, V0, mu, kappa, theta, heston_sigma, rho, time_horizon, steps, simulations)

    heston_final = (
    heston_paths[-1, :]
    )

    gbm_final = gbm_paths[-1, :]

    heston_final = heston_paths[-1, :]
#KS Test
    gbm_returns = (
    gbm_final / IAP_stock - 1
    )
    heston_returns = (
    heston_final / IAP_stock - 1
    )
    ks_stat, ks_p = ks_2samp(
    gbm_returns,
    heston_returns
    )

    results.append({
        "Stock": Stock,
        "KS_Statistic": ks_stat,
        "KS_p_value": ks_p
    })

results_df = pd.DataFrame(results)

KS_test = results_df[[
    "Stock",
    "KS_Statistic",
    "KS_p_value"
]].copy()

KS_test["KS_p_Holm"] = multipletests(
    KS_test["KS_p_value"],
    alpha=0.05,
    method="holm"
)[1]

KS_test["KS_holm_Significant"] = (
    KS_test["KS_p_Holm"] < 0.05
)

KS_test = KS_test.sort_values(
    "KS_Statistic",
    ascending = False
)

display(KS_test)

After applying the Holm correction across the 20 stock-level K-S tests, 7 of the 20 stocks retained statistically significant differences between their GBM and Heston simulated terminal-return distributions. This indicates that, for approximately 35% of the sample, stochastic volatility materially altered the overall return distribution relative to GBM. However, for the remaining 65%, the K-S test did not detect a statistically significant distributional difference after correcting for multiple comparisons.

Important Note: The p-value obtained is heavily influenced by the 10,000 simulated observations, while the K-S Test Statistic is not. Therefore data is sorted by the K-S Test Statistic and give more weight to it in our interpretation of results, rather then the p-value. 

### Wasserstein distance

In [ ]:
#| echo = false

np.random.seed(43)
results = []

for Stock in stocks:

    stock_data = yf.download(
        Stock,
        period="5y",
        progress=False
    )

    if isinstance(stock_data.columns, pd.MultiIndex):
        close_prices = stock_data["Close"][Stock]
    else:
        close_prices = stock_data["Close"]

    close_prices = close_prices.dropna()

    IAP_stock = float(close_prices.iloc[-1])

    log_returns = np.log(
        close_prices / close_prices.shift(1)
    ).dropna()

    # GBM Parameters

    gbm_sigma = log_returns.std() * np.sqrt(steps)

#Heston Parameters

    rolling_variance = (log_returns.rolling(window=21).var()* steps).dropna()

    V0 = float(rolling_variance.iloc[-1])

#Theta and Kappa Estimates
    V_t = rolling_variance.iloc[:-1].values
    V_next = rolling_variance.iloc[1:].values
    delta_V = V_next - V_t
    X = np.column_stack([np.ones(len(V_t)), V_t])

    beta = np.linalg.lstsq(X, V_next, rcond=None)[0]

    dt = 1 / steps

    intercept = beta[0]
    slope = beta[1]
    


    if 0 < slope < 1:

        kappa = -np.log(slope) / dt

        theta = intercept / (1-slope)

    else:

        theta = float(rolling_variance.mean())

        kappa = 2.0

    if not np.isfinite(theta) or theta <= 0:
        theta = float(rolling_variance.mean())

    if not np.isfinite(kappa) or kappa <= 0:
        kappa = 2.0

# Sigma Estimates
    delta_V = V_next - V_t

    expected_delta_V = (kappa* (theta - V_t)* dt)

    variance_residuals = (delta_V - expected_delta_V)

    valid = V_t > 1e-10

    standardized_variance_shocks = (variance_residuals[valid] 
                                    / np.sqrt(V_t[valid] * dt))

    heston_sigma = float(np.std(standardized_variance_shocks,ddof=1))

    if not np.isfinite(heston_sigma) or heston_sigma <= 0:
        heston_sigma = 0.01

#rho Estimate 
    aligned_returns = (log_returns.loc[rolling_variance.index].iloc[1:].values)
    aligned_returns = aligned_returns[valid]
    V_for_returns = V_t[valid]

    stock_shocks = (aligned_returns - mu * dt) / np.sqrt(V_for_returns * dt)

    variance_shocks = (variance_residuals[valid]/ 
                    (heston_sigma * np.sqrt(V_for_returns * dt)))

    rho = float(np.corrcoef(stock_shocks, variance_shocks)[0, 1])

    if not np.isfinite(rho):
        rho = 0.0

    rho = np.clip(rho,-0.999,0.999)

# GBM Sim
    gbm_paths = GBM(IAP_stock, mu, gbm_sigma, time_horizon, steps, simulations)

    gbm_final = (
        gbm_paths[-1, :]
    )

#Heston Sim
    heston_paths, heston_variances = Heston(IAP_stock, V0, mu, kappa, theta, heston_sigma, rho, time_horizon, steps, simulations)

    heston_final = (
    heston_paths[-1, :]
    )

    gbm_final = gbm_paths[-1, :]

    heston_final = heston_paths[-1, :]
#Wasserstein Test
    gbm_returns = (
    gbm_final / IAP_stock - 1
        )
    heston_returns = (
    heston_final / IAP_stock - 1
        )
    wasserstein_dist = wasserstein_distance(
        gbm_returns,
        heston_returns
    )
    results.append({
        "Stock": Stock,
        "Wasserstein_Distance": wasserstein_dist
    })
results_df = pd.DataFrame(results)

Wasserstein_test = results_df[[
    "Stock",
    "Wasserstein_Distance"
]].copy()

Wasserstein_ranked = Wasserstein_test.sort_values(
    "Wasserstein_Distance",
    ascending = False
)

display(Wasserstein_ranked)

Wasserstein distance was used to quantify the magnitude of divergence between the GBM and Heston simulated terminal-return distributions. The largest distances were observed for MSFT (0.103), LRCX (0.102), AMZN (0.076), and DORM (0.076), corresponding to average distributional displacements of approximately 10.3, 10.2, 7.6, and 7.6 percentage points respectively. These stocks were also among those identified as significantly different by the K-S test, suggesting that the observed differences were not only statistically detectable but also economically meaningful in magnitude

Based on these comparison analysis we can see a pattern of specific stocks producing very similar GBM/Heston distributions while others diverge strongly. Based on this we therefore need to investigate which specific effect is causing this diveragence between the models. As we now know why stocks diverage and which ones dont, we can look at the differences between the stocks and which data influences the model, to find the variable behind it. 

## Compare Both Models to True results of stock from September 1st 2025 to September 1st 2026 based on 5 years prior

In [ ]:
#| echo = false
def model_comparison(Stock): 

    np.random.seed(43)

    time_horizon = 1
    steps = 252
    simulations = 10000


    #Simulation data time period
    calibration_start = "2020-09-01"
    calibration_end = "2025-09-01"

    #True time period of stock 
    test_start = "2025-09-01"
    test_end = "2026-09-02"

    #GBM Model
    def GBM(IAP, MU, gbm_sigma, T, n, Paths):
        dt = T / n
        #Variables
        drift = (MU - 0.5 * gbm_sigma**2) * dt
        diffusion = gbm_sigma * np.sqrt(dt)
        Z = np.random.standard_normal(size = (n, Paths))
        #Variables combined
        Path_mult = np.exp(drift + diffusion * Z)
        #Added array of 1's to be replaced with real values by number of n 
        array_fill = np.ones((1, Paths))
        #variable combination into array
        array_full = np.vstack([array_fill, Path_mult])
        paths = IAP * np.cumprod(array_full, axis = 0)

        return paths

    #Heston Model
    def Heston(IAP, V0, r, kappa, theta, heston_sigma, rho, T, n, sims):
        dt = T / n
        S = np.zeros((n + 1, sims))
        V = np.zeros((n + 1, sims))
        S[0] = IAP
        V[0] = V0

        for t in range(1, n + 1):
            Z1 = np.random.normal(0,1,sims)
            Z2 = np.random.normal(0,1,sims)
            W_S = Z1
            W_V = rho * Z1 + np.sqrt(1 - rho**2) * Z2

            V_prev_truncated = np.maximum(V[t-1], 0)

            V[t] = np.maximum(
                V[t-1]
                + kappa * (theta - V_prev_truncated) * dt
                + heston_sigma * np.sqrt(V_prev_truncated * dt) * W_V,
                0)
            S[t] = S[t-1] * np.exp(
                (mu - 0.5 * V_prev_truncated) * dt
                + np.sqrt(V_prev_truncated * dt) * W_S
    )
        return S, V

    #Calibration Data

    print(
        f"Calibrating {Stock} using"
        f"{calibration_start} to {calibration_end}..."
    )

    calibration_data = yf.download(
        Stock,
        start=calibration_start,
        end=calibration_end,
        progress=False,
        auto_adjust=True
    )

    if isinstance(calibration_data.columns, pd.MultiIndex):
        close_prices = (calibration_data["Close"][Stock])
    else:
        close_prices = (calibration_data["Close"])


    close_prices = close_prices.dropna()    

    IAP_stock = float(close_prices.iloc[-1])

    start_date_actual = (close_prices.index[-1])

    #Returns

    log_returns = np.log(
        close_prices
        / close_prices.shift(1)
    ).dropna()


    daily_mean = (
        log_returns.mean()
    )

    daily_var = (
        log_returns.var()
    )


    mu = (
        daily_mean
        + 0.5 * daily_var
    ) * steps

    # GBM Parameters

    gbm_sigma = log_returns.std() * np.sqrt(steps)

    #Heston Parameters

    rolling_variance = (log_returns.rolling(window=21).var()* steps).dropna()

    V0 = float(rolling_variance.iloc[-1])

    #Theta and Kappa Estimates
    V_t = rolling_variance.iloc[:-1].values
    V_next = rolling_variance.iloc[1:].values
    delta_V = V_next - V_t
    X = np.column_stack([np.ones(len(V_t)), V_t])

    beta = np.linalg.lstsq(X, V_next, rcond=None)[0]

    dt = 1 / steps

    intercept = beta[0]
    slope = beta[1]
        


    if 0 < slope < 1:

        kappa = -np.log(slope) / dt

        theta = intercept / (1-slope)

    else:

        theta = float(rolling_variance.mean())

        kappa = 2.0

    if not np.isfinite(theta) or theta <= 0:
        theta = float(rolling_variance.mean())

    if not np.isfinite(kappa) or kappa <= 0:
        kappa = 2.0

    # Heston Sigma Estimates
    delta_V = V_next - V_t

    expected_delta_V = (kappa* (theta - V_t)* dt)

    variance_residuals = (delta_V - expected_delta_V)

    valid = V_t > 1e-10

    standardized_variance_shocks = (variance_residuals[valid] 
                                        / np.sqrt(V_t[valid] * dt))

    heston_sigma = float(np.std(standardized_variance_shocks,ddof=1))

    if not np.isfinite(heston_sigma) or heston_sigma <= 0:
        heston_sigma = 0.01

    #Heston rho Estimate 
    aligned_returns = (log_returns.loc[rolling_variance.index].iloc[1:].values)
    aligned_returns = aligned_returns[valid]
    V_for_returns = V_t[valid]

    stock_shocks = (aligned_returns - mu * dt) / np.sqrt(V_for_returns * dt)

    variance_shocks = (variance_residuals[valid]/ 
                        (heston_sigma * np.sqrt(V_for_returns * dt)))

    rho = float(np.corrcoef(stock_shocks, variance_shocks)[0, 1])

    if not np.isfinite(rho):
        rho = 0.0

    rho = np.clip(rho,-0.999,0.999)

    #Download Actual test Data

    actual_data = yf.download(
        Stock,
        start=test_start,
        end=test_end,
        progress=False,
        auto_adjust=True)

    if isinstance(actual_data.columns, pd.MultiIndex):
        actual_prices = (actual_data["Close"][Stock])
    else:
        actual_prices = (actual_data["Close"])

    actual_prices = (actual_prices.dropna())

    actual_start = pd.Series(
        [IAP_stock],
        index=[start_date_actual]
    )

    actual_start = pd.concat([
        actual_start,
        actual_prices
    ])

    actual_prices = (
        actual_prices[
            ~actual_prices.index.duplicated(keep='first')
        ].sort_index()
    )

    steps = (
        len(actual_prices) - 1
    )

    print(
        f"Simulation trading steps: {steps}"
    )


    # GBM Sim
    gbm_paths = GBM(IAP_stock, mu, gbm_sigma, time_horizon, steps, simulations)

    gbm_final = (
        gbm_paths[-1, :]
    )

    #Heston Sim
    heston_paths, heston_variances = Heston(IAP_stock, V0, mu, kappa, theta, heston_sigma, rho, time_horizon, steps, simulations)

    heston_final = (
    heston_paths[-1, :]
    )

    actual_final = float(
        actual_prices.iloc[-1]
    )
    #Parameters Outputs: 
    print(
        "\n--- Out-of-Sample Model Test ---"
    )
    print(
        f"Calibration ending price : "
        f"${IAP_stock:.2f}"
    )
    print(
        f"Actual ending price      : "
        f"${actual_final:.2f}"
    )
    print(
        f"GBM median forecast      : "
        f"${np.median(gbm_final):.2f}"
    )
    print(
        f"Heston median forecast   : "
        f"${np.median(heston_final):.2f}"
    )
    print(
        f"GBM 5%-95% interval      : "
        f"${np.percentile(gbm_final, 5):.2f} - "
        f"${np.percentile(gbm_final, 95):.2f}"
    )
    print(
        f"Heston 5%-95% interval   : "
        f"${np.percentile(heston_final, 5):.2f} - "
        f"${np.percentile(heston_final, 95):.2f}"
    )
    print(
        "\n--- Calibrated Parameters ---"
    )
    print(
        f"mu           = {mu:.4%}"
    )
    print(
        f"GBM sigma    = {gbm_sigma:.4%}"
    )
    print(
        f"Heston V0    = {V0:.4f}"
    )
    print(
        f"Heston theta = {theta:.4f}"
    )
    print(
        f"Heston kappa = {kappa:.4f}"
    )
    print(
        f"Heston sigma = {heston_sigma:.4f}"
    )
    print(
        f"Heston rho   = {rho:.4f}"
    )

    display_paths = 200

    # Use actual dates for simulation x-axis
    simulation_dates = actual_prices.index

    fig = plt.figure(
        figsize=(18, 10)
    )

    gs = fig.add_gridspec(
        2,
        3,
        width_ratios=[1.5, 1, 1.5]
    )


#PLOTS

    ax1 = fig.add_subplot(
        gs[0, 0]
    )

    for i in range(display_paths):

        ax1.plot(
            simulation_dates,
            gbm_paths[:, i],
            linewidth=0.6,
            alpha=0.25
        )


    ax1.plot(
        simulation_dates,
        np.median(
            gbm_paths,
            axis=1
        ),
        linewidth=2,
        label="GBM Median",
        color="red"
    )


    ax1.plot(
        actual_prices.index,
        actual_prices.values,
        linewidth=2.5,
        label=f"Actual {Stock}",
        color="darkorange"
    )


    ax1.set_title(
        f"GBM Simulation vs Actual {Stock}"
    )

    ax1.set_ylabel(
        "Stock Price ($)"
    )

    ax1.grid(
        True,
        linestyle="--",
        alpha=0.3
    )

    ax1.legend()


    # ============================================================
    # GBM TERMINAL DISTRIBUTION
    # ============================================================

    ax2 = fig.add_subplot(
        gs[0, 1]
    )

    ax2.hist(
        gbm_final,
        bins=40,
        alpha=0.7,
        edgecolor="black"
    )


    ax2.axvline(
        IAP_stock,
        linestyle="--",
        linewidth=2,
        label=f"Start ${IAP_stock:.2f}",
        color="green"
    )

    ax2.axvline(
        np.median(gbm_final),
        linewidth=2,
        label=(
            f"GBM Median "
            f"${np.median(gbm_final):.2f}"
        ),
        color="red"
    )

    ax2.axvline(
        actual_final,
        linewidth=2.5,
        label=(
            f"Actual "
            f"${actual_final:.2f}"
        ),
        color="darkorange"
    )


    ax2.set_title(
        "GBM Terminal Distribution"
    )

    ax2.set_xlabel(
        "Ending Price ($)"
    )

    ax2.set_ylabel(
        "Count"
    )

    ax2.grid(
        True,
        linestyle="--",
        alpha=0.3
    )

    ax2.legend()


    # ============================================================
    # HESTON PATHS
    # ============================================================

    ax3 = fig.add_subplot(
        gs[1, 0]
    )

    for i in range(display_paths):

        ax3.plot(
            simulation_dates,
            heston_paths[:, i],
            linewidth=0.6,
            alpha=0.25
        )


    ax3.plot(
        simulation_dates,
        np.median(
            heston_paths,
            axis=1
        ),
        linewidth=2,
        label="Heston Median",
        color="red"
    )


    ax3.plot(
        actual_prices.index,
        actual_prices.values,
        linewidth=2.5,
        label=f"Actual {Stock}",
        color="darkorange"
    )


    ax3.set_title(
        f"Heston Simulation vs Actual {Stock}"
    )

    ax3.set_xlabel(
        "Date"
    )

    ax3.set_ylabel(
        "Stock Price ($)"
    )

    ax3.grid(
        True,
        linestyle="--",
        alpha=0.3
    )

    ax3.legend()


    # ============================================================
    # HESTON TERMINAL DISTRIBUTION
    # ============================================================

    ax4 = fig.add_subplot(
        gs[1, 1]
    )

    ax4.hist(
        heston_final,
        bins=40,
        alpha=0.7,
        edgecolor="black"
    )


    ax4.axvline(
        IAP_stock,
        linestyle="--",
        linewidth=2,
        label=f"Start ${IAP_stock:.2f}",
        color="green"
    )

    ax4.axvline(
        np.median(heston_final),
        linewidth=2,
        label=(
            f"Heston Median "
            f"${np.median(heston_final):.2f}"
        ),
        color="red"
    )

    ax4.axvline(
        actual_final,
        linewidth=2.5,
        label=(
            f"Actual "
            f"${actual_final:.2f}"
        ),
        color="darkorange"
    )


    ax4.set_title(
        "Heston Terminal Distribution"
    )

    ax4.set_xlabel(
        "Ending Price ($)"
    )

    ax4.set_ylabel(
        "Count"
    )

    ax4.grid(
        True,
        linestyle="--",
        alpha=0.3
    )

    ax4.legend()


    # ============================================================
    # ACTUAL STOCK MOVEMENT
    # ============================================================

    ax5 = fig.add_subplot(
        gs[:, 2]
    )

    ax5.plot(
        actual_prices.index,
        actual_prices.values,
        linewidth=2.5,
        color="darkorange"
    )

    ax5.axhline(
        IAP_stock,
        linestyle="--",
        linewidth=1.5,
        label=(
            f"Start ${IAP_stock:.2f}"
        ),
        color="green"
    )

    ax5.scatter(
        actual_prices.index[-1],
        actual_final,
        s=60,
        zorder=5,
        label=(
            f"End ${actual_final:.2f}"
        )
    )


    ax5.set_title(
        f"Actual {Stock} Movement\n"
        "Sep 2025 – Sep 2026"
    )

    ax5.set_xlabel(
        "Date"
    )

    ax5.set_ylabel(
        "Stock Price ($)"
    )

    ax5.grid(
        True,
        linestyle="--",
        alpha=0.3
    )

    ax5.legend()


    plt.tight_layout()

    plt.show()

    GBM_error = (
        np.abs(np.median(gbm_final) - actual_final)/ actual_final)
    Heston_error = (
        np.abs(np.median(heston_final) - actual_final)/ actual_final)
    gbm_percentile = (
        np.mean(gbm_final <= actual_final)
        * 100
    )
    heston_percentile = (
        np.mean(heston_final <= actual_final)
        * 100
    )
    gbm_interval_width = (
        np.percentile(gbm_final, 95)
        - np.percentile(gbm_final, 5)
    )
    heston_interval_width = (
        np.percentile(heston_final, 95)
        - np.percentile(heston_final, 5)
    )
    print(
        f"GBM Median Error: {GBM_error:.2%}"
    )
    print(
        f"Heston Median Error: {Heston_error:.2%}"
    )
    print(
        f"GBM Actual Percentile: {gbm_percentile:.2f}%"
    )
    print(
        f"Heston Actual Percentile: {heston_percentile:.2f}%"
    )
    print(
        f"GBM 5%-95% Interval Width: ${gbm_interval_width:.2f}"
    )
    print(
        f"Heston 5%-95% Interval Width: ${heston_interval_width:.2f}"
    )


In [ ]:
model_comparison("MSFT")

Both models overestimated Microsoft’s one-year terminal price because both inherited a strongly positive historical drift from the calibration period. GBM produced a slightly more accurate median terminal forecast, while Heston generated a narrower predictive distribution that still contained the realized outcome. The difference between the two distributions appears to arise primarily from Heston’s mean-reverting stochastic variance process rather than the leverage effect, as the estimated price-volatility correlation was close to zero.

In [ ]:
model_comparison("MU")

Pepsi exhibited one of the smallest Wasserstein distances between GBM and Heston. This appears to be explained by the close agreement between GBM’s constant variance and Heston’s estimated initial and long-run variance. In addition, Heston’s relatively fast mean reversion and weak price-volatility correlation limited the extent to which stochastic volatility altered the simulated distribution. As a result, both models produced nearly identical medians, prediction intervals, and realized-price percentiles.